# FlightAI — multimodal chat (text, image, audio)

Week 2 Day 5: custom Gradio **Blocks** UI. The model looks up ticket prices (SQLite tool), speaks the reply (`gpt-4o-mini-tts`), and generates a pop-art city image (`gpt-image-1-mini`) when a destination is looked up.

Needs `OPENAI_API_KEY` in `.env`. Image generation costs a few cents per call.

In [ ]:
import base64
import json
import os
import sqlite3
from io import BytesIO

import gradio as gr
from dotenv import load_dotenv
from openai import OpenAI
from PIL import Image

load_dotenv(override=True)
openai = OpenAI()
MODEL = "gpt-4.1-mini"

system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False,
    },
}
tools = [{"type": "function", "function": price_function}]

In [ ]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    conn.execute(
        "CREATE TABLE IF NOT EXISTS prices (city TEXT PRIMARY KEY, price REAL)"
    )
    conn.commit()


def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        conn.execute(
            "INSERT INTO prices (city, price) VALUES (?, ?) ON CONFLICT(city) DO UPDATE SET price = ?",
            (city.lower(), price, price),
        )
        conn.commit()


for city, price in {
    "london": 799,
    "paris": 899,
    "tokyo": 1420,
    "sydney": 2999,
    "hyderabad": 499,
}.items():
    set_ticket_price(city, price)


def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        row = conn.execute(
            "SELECT price FROM prices WHERE city = ?", (city.lower(),)
        ).fetchone()
    return (
        f"Ticket price to {city} is ${row[0]}"
        if row
        else "No price data available for this city"
    )

In [ ]:
def artist(city):
    """Generate a 1024x1024 pop-art city image; API returns base64 PNG."""
    image_response = openai.images.generate(
        model="gpt-image-1-mini",
        prompt=(
            f"An image representing a vacation in {city}, showing tourist spots "
            f"and everything unique about {city}, in a vibrant pop-art style"
        ),
        size="1024x1024",
        n=1,
    )
    image_data = base64.b64decode(image_response.data[0].b64_json)
    return Image.open(BytesIO(image_data))


def talker(message):
    """TTS bytes for Gradio Audio (autoplay). Try voice alloy or coral."""
    response = openai.audio.speech.create(
        model="gpt-4o-mini-tts",
        voice="onyx",
        input=message,
    )
    return response.content

In [ ]:
def handle_tool_calls_and_return_cities(message):
    """Run each tool call and collect cities so we can generate one image."""
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get("destination_city")
            cities.append(city)
            responses.append(
                {
                    "role": "tool",
                    "content": get_ticket_price(city),
                    "tool_call_id": tool_call.id,
                }
            )
    return responses, cities


def chat(history):
    # Gradio already appended the user turn; history is the full messages list.
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = openai.chat.completions.create(
        model=MODEL, messages=messages, tools=tools
    )
    cities = []
    image = None

    while response.choices[0].finish_reason == "tool_calls":
        assistant_msg = response.choices[0].message
        tool_results, cities = handle_tool_calls_and_return_cities(assistant_msg)
        messages.append(assistant_msg)
        messages.extend(tool_results)
        response = openai.chat.completions.create(
            model=MODEL, messages=messages, tools=tools
        )

    reply = response.choices[0].message.content
    history += [{"role": "assistant", "content": reply}]
    voice = talker(reply)
    if cities:
        image = artist(cities[0])
    return history, voice, image


def put_message_in_chatbot(message, history):
    """Clear the textbox and show the user message immediately, then chat() runs."""
    return "", history + [{"role": "user", "content": message}]


with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500, type="messages")
        image_output = gr.Image(height=500, interactive=False)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

    # submit: paint user bubble first, then fill assistant + audio + image.
    message.submit(
        put_message_in_chatbot,
        inputs=[message, chatbot],
        outputs=[message, chatbot],
    ).then(chat, inputs=chatbot, outputs=[chatbot, audio_output, image_output])

ui.launch(inbrowser=True)